# Conversational AI - alias Chatbot!

Pada modul ini, kita akan fokus membangun asisten AI percakapan (Chatbot) yang interaktif. 

Kita akan menggunakan dua pustaka utama:
1. **OpenAI API**: Sebagai "otak" dari chatbot kita, menggunakan model bahasa (LLM) untuk memahami dan menghasilkan teks.
2. **Gradio**: Sebuah pustaka Python fantastis yang memungkinkan kita membuat antarmuka pengguna (User Interface / UI) berbasis web hanya dengan beberapa baris kode, tanpa perlu mengerti HTML/CSS/JavaScript.

Di akhir *notebook* ini, kita akan membangun chatbot yang memiliki **Persona** khusus (sebagai asisten toko pakaian) yang mampu mengingat riwayat percakapan sebelumnya.

In [27]:
# MENGIMPOR PUSTAKA YANG DIBUTUHKAN

import os
from dotenv import load_dotenv # Digunakan untuk memuat variabel lingkungan dari file .env
from openai import OpenAI      # Pustaka resmi OpenAI untuk memanggil API
import gradio as gr            # Pustaka antarmuka web untuk membuat UI Chatbot

print("Pustaka berhasil diimpor!")

Pustaka berhasil diimpor!


## Memuat Kunci API (API Key)

Untuk menggunakan layanan OpenAI, kita memerlukan API Key. Sangat **TIDAK DISARANKAN** untuk menulis API Key langsung di dalam kode (*hardcode*) karena alasan keamanan. 

Sebagai praktik terbaik, kita menggunakan pustaka `python-dotenv` untuk memuat variabel lingkungan dari sebuah file bernama `.env` yang berada di folder yang sama.

Format isi file `.env` Anda seharusnya seperti ini:
`OPENAI_API_KEY=sk-xxxxxxx...`

In [28]:
# Memuat variabel lingkungan dari file .env (jika ada)
# Parameter override=True memastikan variabel di file .env menimpa variabel sistem yang ada
load_dotenv(override=True)

# Mengambil nilai kunci API
openai_api_key = os.getenv('OPENAI_API_KEY')

# Mengecek dan mencetak 8 karakter pertama untuk memastikan kunci berhasil dimuat (debugging)
if openai_api_key:
    print(f"OpenAI API Key berhasil dimuat dan diawali dengan: {openai_api_key[:8]}...")
else:
    print("PERINGATAN: OpenAI API Key tidak ditemukan. Pastikan file .env sudah diatur.")

OpenAI API Key berhasil dimuat dan diawali dengan: sk-proj-...


## Inisialisasi Klien OpenAI dan Konfigurasi Model

Selanjutnya, kita akan membuat objek klien dari OpenAI yang akan digunakan untuk mengirim *prompt* ke server mereka. Kita juga akan mendefinisikan model apa yang ingin kita gunakan. Pada contoh ini, kita menggunakan `gpt-4o-mini` (atau model mini setara lainnya) karena lebih cepat dan efisien untuk tugas percakapan dasar.

In [29]:
# Inisialisasi klien OpenAI. Klien ini secara otomatis akan mencari variabel lingkungan OPENAI_API_KEY.
openai = OpenAI()

# Mendefinisikan model yang akan digunakan. 
# Catatan: Sesuaikan nama model jika diperlukan (misal: 'gpt-4o-mini' atau 'gpt-3.5-turbo')
MODEL = 'gpt-4o-mini' 

print(f"Klien OpenAI siap menggunakan model: {MODEL}")

Klien OpenAI siap menggunakan model: gpt-4o-mini


## Menyiapkan *System Message*

Dalam arsitektur model *chat* seperti GPT-4, terdapat pesan khusus yang disebut **System Message** (Pesan Sistem). Pesan ini tidak terlihat oleh pengguna akhir, tetapi berfungsi sebagai "instruksi fundamental" bagi AI untuk menentukan identitas, nada bicara, aturan, dan batasan perilakunya.

Untuk tahap awal, kita buat instruksi sederhana.

In [30]:
# Kita mendefinisikan variabel global untuk pesan sistem.
# Kita akan sering mengubah nilai ini nanti untuk melihat bagaimana AI mengubah perilakunya.

system_message = "You are a helpful assistant"

## Membangun Fungsi *Callback* untuk Chat

Untuk menggunakan komponen `gr.ChatInterface` di Gradio, kita harus membuat sebuah fungsi (sering disebut fungsi *callback*).

Fungsi ini **wajib** menerima dua argumen:
1. `message` (string): Pesan terbaru yang diketik oleh pengguna.
2. `history` (list): Riwayat percakapan sebelumnya. 
   *(Karena kita menghindari `type="messages"`, bentuk bawaan history di Gradio adalah list yang berisi pasangan pesan `[[user_msg_1, bot_msg_1], [user_msg_2, bot_msg_2]]`)*.

Fungsi ini harus mengembalikan (*return*) string yang merupakan balasan dari chatbot. Mari kita mulai dengan fungsi *dummy* (pura-pura) terlebih dahulu.

In [33]:
# Fungsi Chat Dummy versi 1: Selalu membalas dengan kata yang sama
def chat_dummy_1(message, history):
    return "Hello"

In [34]:
# Menjalankan antarmuka Gradio. 
# Perhatikan bahwa kita TIDAK menggunakan type="messages" agar kompatibel dengan versi Anda.

print("Menjalankan Chatbot Dummy 1...")
gr.ChatInterface(fn=chat_dummy_1).launch()

Menjalankan Chatbot Dummy 1...
* Running on local URL:  http://127.0.0.1:7886
* To create a public link, set `share=True` in `launch()`.


Coba ketikkan sesuatu di antarmuka di atas. Apapun yang Anda ketik, bot hanya akan menjawab "Hello". 

Sekarang, mari kita buat fungsi *dummy* kedua untuk memahami bagaimana Gradio mengirimkan format `history` ke dalam fungsi kita. Ini sangat penting untuk debugging.

In [37]:
# Fungsi Chat Dummy versi 2: Memperlihatkan format pesan dan riwayat
def chat_dummy_2(message, history):
    # Kita akan mencetak riwayat agar bisa melihat bentuk struktur datanya
    return f"Anda baru saja berkata: '{message}'.\nRiwayat saat ini berbentuk: {history}\nNamun saya tetap bilang: Hello!"

In [38]:
print("Menjalankan Chatbot Dummy 2...")
gr.ChatInterface(fn=chat_dummy_2).launch()

Menjalankan Chatbot Dummy 2...
* Running on local URL:  http://127.0.0.1:7888
* To create a public link, set `share=True` in `launch()`.


## Menghubungkan ke OpenAI API (Tanpa *Streaming*)

Sekarang saatnya menghubungkan UI kita dengan otak AI sungguhan.
Karena kita tidak menggunakan `type="messages"`, kita harus mengubah format `history` bawaan Gradio (yaitu `[[user, bot], [user, bot]]`) menjadi format kamus yang dipahami oleh OpenAI API (yaitu `[{"role": "...", "content": "..."}, ...]`).

Langkah-langkah di fungsi ini:
1. Masukkan pesan sistem sebagai pesan pertama.
2. Ekstrak pesan pengguna dan asisten dari riwayat Gradio, lalu ubah ke format OpenAI.
3. Tambahkan pesan terbaru pengguna di urutan terakhir.
4. Kirim ke API dan ambil balasannya.

In [39]:
def chat_real(message, history):
    # 1. Mulai daftar pesan dengan System Message
    messages = [{"role": "system", "content": system_message}]
    
    # 2. Parsing riwayat dari Gradio ke format OpenAI
    # history bentuknya: [["Halo", "Hai ada yang bisa dibantu?"], ["Siapa kamu?", "Saya AI."]]
    for user_text, assistant_text in history:
        messages.append({"role": "user", "content": user_text})
        messages.append({"role": "assistant", "content": assistant_text})
        
    # 3. Tambahkan pesan pengguna yang paling baru
    messages.append({"role": "user", "content": message})
    
    # 4. Panggil OpenAI API (Proses sinkronus, menunggu sampai jawaban selesai)
    response = openai.chat.completions.create(
        model=MODEL, 
        messages=messages
    )
    
    # 5. Kembalikan isi konten dari jawaban AI
    return response.choices[0].message.content

In [59]:
print("Menjalankan Chatbot Real (Non-Streaming)...")
gr.ChatInterface(fn=chat_real).launch()

Menjalankan Chatbot Real (Non-Streaming)...
* Running on local URL:  http://127.0.0.1:7898
* To create a public link, set `share=True` in `launch()`.


## Menambahkan Efek *Streaming* (Mengetik Karakter demi Karakter)

Menunggu API menghasilkan teks panjang bisa terasa lama. Pengalaman pengguna (UX) akan jauh lebih baik jika teks muncul sedikit demi sedikit seperti sedang diketik. 

Di OpenAI API, kita bisa mengaktifkan `stream=True`. 
Di sisi Python, kita mengubah fungsi kita menjadi sebuah **Generator** menggunakan kata kunci `yield`. Setiap kali OpenAI mengirim potongan kata (*chunk*), Gradio akan langsung menampilkannya ke layar.

In [51]:
def chat_stream(message, history):
    # 1. Mulai dengan System Message
    messages = [{"role": "system", "content": system_message}]
    
    # 2. Parsing history yang berisi objek atau dictionary dari Gradio
    for msg in history:
        # Pengecekan aman: jika Gradio mengirimkan dictionary
        if isinstance(msg, dict):
            messages.append({"role": msg["role"], "content": msg["content"]})
        # Jika Gradio mengirimkan objek (ChatMessage)
        else:
            messages.append({"role": msg.role, "content": msg.content})
            
    # 3. Tambahkan pesan pengguna yang terbaru
    messages.append({"role": "user", "content": message})

    # 4. Panggil API dengan parameter stream=True
    stream = openai.chat.completions.create(
        model=MODEL, 
        messages=messages, 
        stream=True
    )
    
    # 5. Kumpulkan potongan teks (chunks) secara bertahap
    partial_response = ""
    for chunk in stream:
        partial_response += chunk.choices[0].delta.content or ''
        yield partial_response

In [52]:
print("Menjalankan Chatbot Real (Streaming)...")
gr.ChatInterface(fn=chat_stream).launch()

Menjalankan Chatbot Real (Streaming)...
* Running on local URL:  http://127.0.0.1:7894
* To create a public link, set `share=True` in `launch()`.


## Memberikan "Persona" Tingkat Lanjut (*One-Shot Prompting*)

Salah satu kekuatan utama LLM adalah kemampuannya mengambil peran yang kita berikan lewat `System Message`. Teknik memberikan contoh secara langsung dalam *prompt* sistem ini sering disebut **One-Shot Prompting**.

Mari kita ubah AI kita menjadi penjaga toko pakaian yang persuasif!

In [53]:
# Mendefinisikan ulang pesan sistem dengan persona yang detail
system_message = (
    "You are a helpful assistant in a clothes store. You should try to gently encourage "
    "the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. "
    "For example, if the customer says 'I'm looking to buy a hat', "
    "you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.' "
    "Encourage the customer to buy hats if they are unsure what to get."
)

In [54]:
# Meluncurkan kembali fungsi stream dengan pesan sistem yang baru
print("Menjalankan Chatbot Toko Pakaian...")
gr.ChatInterface(fn=chat_stream).launch()

Menjalankan Chatbot Toko Pakaian...
* Running on local URL:  http://127.0.0.1:7895
* To create a public link, set `share=True` in `launch()`.


Anda dapat langsung mengubah dan menambahkan aturan pada `System Message` di tengah proses pengembangan untuk menguji keluwesan AI. Mari kita tambahkan aturan baru tentang sepatu.

In [55]:
# Menambahkan aturan tambahan ke variabel system_message yang sudah ada
system_message += (
    "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, "
    "but remind the customer to look at hats!"
)

In [56]:
# Meluncurkan lagi. Cobalah bertanya soal sepatu!
print("Menjalankan Chatbot Toko Pakaian (Update Aturan Sepatu)...")
gr.ChatInterface(fn=chat_stream).launch()

Menjalankan Chatbot Toko Pakaian (Update Aturan Sepatu)...
* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.


## Pesan Sistem Dinamis (Berdasarkan Konteks Pesan Pengguna)

Terkadang kita tidak ingin memberikan instruksi yang terlalu panjang di awal jika tidak diperlukan. Kita bisa menyisipkan aturan logika (seperti `if-else` pada Python) untuk menambahkan informasi spesifik ke dalam pesan sistem *hanya* jika pengguna menanyakan kata kunci tertentu (misalnya, menanyakan soal ikat pinggang / *belt*).

In [57]:
def chat_dynamic_system(message, history):
    # Kita buat variabel pesan sistem yang bersifat sementara untuk fungsi ini
    relevant_system_message = system_message
    
    # Aturan Dinamis: Jika kata 'belt' (ikat pinggang) terdeteksi dalam pesan pengguna
    if 'belt' in message.lower():
        relevant_system_message += (
            " The store does not sell belts; if you are asked for belts, "
            "be sure to point out other items on sale."
        )
    
    # Parsing history
    messages = [{"role": "system", "content": relevant_system_message}]
    for user_text, assistant_text in history:
        messages.append({"role": "user", "content": user_text})
        messages.append({"role": "assistant", "content": assistant_text})
        
    messages.append({"role": "user", "content": message})

    # Memanggil model
    stream = openai.chat.completions.create(
        model=MODEL, 
        messages=messages, 
        stream=True
    )

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [58]:
print("Menjalankan Chatbot dengan Logika Sistem Dinamis...")
gr.ChatInterface(fn=chat_dynamic_system).launch()

Menjalankan Chatbot dengan Logika Sistem Dinamis...
* Running on local URL:  http://127.0.0.1:7897
* To create a public link, set `share=True` in `launch()`.


## Kesimpulan: Aplikasi Bisnis

Asisten Percakapan (Chatbots) merupakan kasus penggunaan yang sangat masif untuk AI Generatif. Model-model terbaru sangat ahli dalam mengelola percakapan dengan nuansa yang kompleks. 

Pustaka seperti Gradio memudahkan kita menyediakan UI, sementara teknik *System Prompting* memastikan AI tidak keluar jalur dan memberikan respons sesuai ekspektasi bisnis kita (misalnya melakukan penawaran promosi *(upselling)* secara natural).

**Langkah selanjutnya:** Pikirkan bagaimana Anda bisa mengimplementasikan Asisten AI untuk ranah minat atau proyek Anda. Anda bisa memodifikasi *system prompt* di atas untuk bertindak sebagai asisten *coding* ahli atau tutor.